### Baby script for fixing labels on mismatched trials

In [ ]:
import pandas as pd
import numpy as np 
from matplotlib import pyplot as plt
import matplotlib

In [ ]:
# load
cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] 
true_labels = pd.read_excel("final_scores_16042026_v4.xlsx")

# files to relabel 
features_all = pd.read_pickle("training_features_19032026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap'})

trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 

In [ ]:
true_labels.head()

Relabeling the files

In [ ]:
key_cols = ['Subject', 'Nap', 'Triggers_Order_Nap']

# making true labels df match the organization of the mismatch column df 
true_labels_wide = (
    true_labels[true_labels['Muscle_type'].isin(['Zygo', 'Corr'])]
    .pivot_table(
        index=key_cols,
        columns='Muscle_type',
        values='Contraction_number',
        aggfunc='first'   
    )
    .reset_index()
    .rename(columns={
        'Zygo': 'Num_Contractions_Zygo',
        'Corr': 'Num_Contractions_Corr'
    })
)


In [ ]:
df_all_temp = df_all[df_all['Num_Contractions_Corr_y'].notna()]
df_all_temp[df_all_temp['Subject'] == 'NL03JV']
#df_all_temp.head(50)

In [ ]:
true_labels_wide.head()
df_all = features_all.merge(true_labels_wide, on=key_cols, how='left')

# checking mismatches 
zygo_mismatch = (
    df_all['Num_Contractions_Zygo_y'].notna() &
    (df_all['Num_Contractions_Zygo_y'] != df_all['Num_Contractions_Zygo_x'])
)

corr_mismatch = (
    df_all['Num_Contractions_Corr_y'].notna() &
    (df_all['Num_Contractions_Corr_y'] != df_all['Num_Contractions_Corr_x'])
)

df_all.loc[zygo_mismatch, 'Num_Contractions_Zygo_x'] = df_all.loc[zygo_mismatch, 'Num_Contractions_Zygo_y'].astype(int)
df_all.loc[corr_mismatch, 'Num_Contractions_Corr_x'] = df_all.loc[corr_mismatch, 'Num_Contractions_Corr_y'].astype(int)
 
changed_rows = df_all[zygo_mismatch | corr_mismatch].copy()
print("Number of rows updated:", len(changed_rows))
print(changed_rows[
    key_cols +
    ['Num_Contractions_Zygo_x', 'Num_Contractions_Corr_x',
     'Num_Contractions_Zygo_y', 'Num_Contractions_Zygo_y']
].head())

# fix df organization 
df_all = df_all.rename(columns={
    'Num_Contractions_Zygo_x': 'Num_Contractions_Zygo',
    'Num_Contractions_Corr_x': 'Num_Contractions_Corr'
}).drop(columns=[
    'Num_Contractions_Zygo_y',
    'Num_Contractions_Corr_y'
])



In [ ]:
df_all.to_pickle("training_features_18042026.pkl")
df_all.to_excel("training_features_18042026.xlsx", index=False)